# CampusReg: shared RAG core

This notebook uses the installable campusreg package in backend/; the PDF extraction, chunking, embeddings, retrieval, and generation code is shared with local use. The starter notebook is preserved as 02_rag_starter_kit_baseline.ipynb. Upload verified official source PDFs before asking policy questions. Synthetic sample policies are intentionally omitted.


## Setup

Run this notebook from a checkout of the CampusReg repository. The setup cell locates the checkout and installs the shared package plus its notebook integrations in editable mode.


In [ ]:
from pathlib import Path
import subprocess
import sys


def _find_checkout_root():
    starts = [Path.cwd(), *Path.cwd().parents]
    content = Path("/content")
    if content.exists():
        starts.extend(path for path in content.iterdir() if path.is_dir())
    for candidate in starts:
        if (candidate / "backend" / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError(
        "CampusReg checkout not found. Check out the repository in the Colab runtime and rerun."
    )


REPO_ROOT = _find_checkout_root()
install_target = f"{REPO_ROOT / 'backend'}[notebook]"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", install_target],
    check=True,
)
print("CampusReg shared package installed from the repository checkout.")


## Configure the Google AI Studio key

In Colab Secrets, add GOOGLE_API_KEY (or the legacy gemma_API_Key). The key is read at runtime and is never displayed or written to a file. For local use, set GOOGLE_API_KEY in the process environment before starting the notebook.


In [ ]:
from campusreg.config import RAGSettings
from campusreg.rag import GoogleGenAIProvider, MultiDocumentRAGEngine, SentenceTransformerEmbedder
from campusreg.storage import SQLiteDocumentStore

settings = RAGSettings.from_env()
embedder = SentenceTransformerEmbedder(settings.embedding_model)
generator = GoogleGenAIProvider.from_env()
store = SQLiteDocumentStore(REPO_ROOT / "data" / "campusreg.sqlite3")
rag = MultiDocumentRAGEngine(settings, embedder, generator, store)
print("Persistent multi-document RAG core ready. No model request has been made yet.")


## Ask questions about official documents

Only upload verified official campus source documents for factual policy answers. Retrieved passages are shown with their source filename, PDF page, and cosine similarity score. The score is a ranking signal, not a probability of correctness.


In [ ]:
import gradio as gr

APP_TITLE = "Campus Regulations Assistant"
APP_DESCRIPTION = "Upload verified official campus PDFs and ask a question grounded in retrieved passages."


def on_upload(file_paths):
    if not file_paths:
        return "No documents uploaded."
    if isinstance(file_paths, (str, Path)):
        file_paths = [file_paths]

    statuses = []
    for file_path in file_paths:
        path = Path(file_path)
        try:
            result = rag.ingest_pdf(path.read_bytes(), path.name)
            record = result.document
            state = "already indexed (duplicate content)" if result.duplicate else "indexed"
            statuses.append(
                f"{record.filename}: {state}; {record.page_count} pages, {record.chunk_count} chunks."
            )
        except Exception as exc:
            statuses.append(f"{path.name}: upload failed ({type(exc).__name__}).")
    return "\n".join(statuses)


def on_ask(question):
    if not question or not question.strip():
        return "Enter a question first.", ""
    try:
        result = rag.answer(question)
    except Exception as exc:
        return f"The request could not be completed ({type(exc).__name__}).", ""

    if result.status.value == "generation_error":
        answer_text = "Generation failed; no policy answer was produced."
    elif result.status.value == "invalid_response":
        answer_text = "The generated response lacked a retrieved source citation; no answer was returned."
    else:
        answer_text = result.answer or "The answer could not be found in the documents."

    sources_text = "\n\n".join(
        f"**{source.citation}** (cosine similarity {source.score:.2f})\n\n> {source.text}"
        for source in result.sources
    )
    return answer_text, sources_text


with gr.Blocks(title=APP_TITLE) as demo:
    gr.Markdown(f"# {APP_TITLE}\n{APP_DESCRIPTION}")
    pdf_input = gr.File(
        label="Official campus PDFs",
        file_types=[".pdf"],
        file_count="multiple",
        type="filepath",
    )
    status_md = gr.Markdown("Upload verified official PDFs to begin.")
    question_box = gr.Textbox(label="Question", placeholder="Ask about the uploaded documents")
    ask_btn = gr.Button("Ask", variant="primary")
    answer_md = gr.Markdown()
    with gr.Accordion("Retrieved evidence", open=True):
        sources_md = gr.Markdown()
    pdf_input.upload(on_upload, inputs=pdf_input, outputs=status_md)
    ask_btn.click(on_ask, inputs=question_box, outputs=[answer_md, sources_md])
    question_box.submit(on_ask, inputs=question_box, outputs=[answer_md, sources_md])

demo.launch(share=False)


## Baseline and limitations

The default configuration keeps the starter values: 120-word chunks, 30-word overlap, top-k 3, sentence-transformers/all-MiniLM-L6-v2, gemma-4-26b-a4b-it, and temperature 0.2. No relevance threshold is enabled unless configured. Generation errors are shown as failures with no policy answer. The notebook does not provide synthetic policy data; scan-heavy PDFs without extractable text are unsupported by this baseline.
